In [ ]:

import pandas as pd
import numpy as np

print("Entorno de ingeniería de datos listo")
print("Versión de Pandas:", pd.__version__)


In [ ]:


import pandas as pd
from pathlib import Path

# Ruta del archivo original
ruta = Path("netflix_titles.csv")

# Verificar que el archivo exista
if ruta.exists():
    df_raw = pd.read_csv(ruta, encoding="latin-1")

    print("Archivo cargado correctamente")
    print("Filas:", df_raw.shape[0])
    print("Columnas:", df_raw.shape[1])

    display(df_raw.head())
else:
    print("No se encontró el archivo CSV.")
    print("Carpeta actual:", Path.cwd())


In [ ]:

# Revisar los nombres de las columnas
print("Columnas del dataset:")
print(df_raw.columns.tolist())

# Identificar columnas completamente vacías
columnas_vacias = df_raw.columns[
    df_raw.isna().all()
].tolist()

print("\nColumnas completamente vacías:")
print(columnas_vacias)

print("\nCantidad de columnas vacías:", len(columnas_vacias))


In [ ]:

# Crear una copia para no modificar los datos originales
df = df_raw.copy()

# Eliminar columnas completamente vacías
df = df.dropna(axis=1, how="all")

# Verificar el resultado
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

display(df.head())


In [ ]:

# 1. Analizar valores nulos por columna
calidad = pd.DataFrame({
    "Valores nulos": df.isna().sum(),
    "Porcentaje nulos": (
        df.isna().mean() * 100
    ).round(2)
})

print("REPORTE DE VALORES NULOS")
display(calidad)

# 2. Revisar duplicados
print("REGISTROS DUPLICADOS")

print(
    "Filas completamente duplicadas:",
    df.duplicated().sum()
)

print(
    "Identificadores show_id duplicados:",
    df.duplicated(subset=["show_id"]).sum()
)

# 3. Revisar tipos de datos
print("TIPOS DE DATOS")
display(df.dtypes)


In [ ]:

# Revisar los tres registros documentados
ids_revisar = ["s5965", "s5966", "s5967"]

registros = df[
    df["show_id"].isin(ids_revisar)
]

display(registros[
    ["show_id", "type", "title", "release_year",
     "country", "listed_in"]
])

# Comprobar si existen en el CSV original
print("Registros encontrados:", len(registros))


In [ ]:

# Revisar los valores únicos de las columnas principales

print("TIPOS DE CONTENIDO")
print(df["type"].value_counts())

print("\nCLASIFICACIONES")
print(df["rating"].value_counts(dropna=False))

print("\nEJEMPLOS DE DURACIÓN")
print(df["duration"].dropna().unique()[:15])

print("\nRANGO DE AÑOS")
print("Mínimo:", df["release_year"].min())
print("Máximo:", df["release_year"].max())


In [ ]:

# Identificar registros con duraciones en la columna rating
ratings_sospechosos = ["74 min", "84 min", "66 min"]

registros_sospechosos = df[
    df["rating"].isin(ratings_sospechosos)
]

display(
    registros_sospechosos[
        ["show_id", "type", "title", "rating", "duration"]
    ]
)

print(
    "Registros sospechosos:",
    len(registros_sospechosos)
)


In [ ]:

# Crear una copia de trabajo
df_clean = df.copy()

# Identificar duraciones ubicadas en rating
mascara = (
    df_clean["rating"].astype("string")
    .str.fullmatch(r"\d+\s+min", na=False)
    & df_clean["duration"].isna()
)

print("Registros a corregir:", mascara.sum())

# Mover el valor a la columna correcta
df_clean.loc[mascara, "duration"] = (
    df_clean.loc[mascara, "rating"]
)

# Dejar rating como valor faltante
df_clean.loc[mascara, "rating"] = None

# Comprobar la corrección
display(
    df_clean.loc[
        mascara,
        ["show_id", "title", "rating", "duration"]
    ]
)


In [ ]:

# Separar duración en valor y unidad
duracion = df_clean["duration"].astype("string").str.extract(
    r"^\s*(\d+)\s+(.+?)\s*$"
)

# Convertir el valor a un tipo numérico
df_clean["duration_value"] = pd.to_numeric(
    duracion[0], errors="coerce"
).astype("Int64")

# Guardar la unidad de duración
df_clean["duration_unit"] = duracion[1].str.strip()

# Comprobar los resultados
display(
    df_clean[
        ["show_id", "type", "duration",
         "duration_value", "duration_unit"]
    ].head(10)
)

print(
    "Duraciones sin valor numérico:",
    df_clean["duration_value"].isna().sum()
)

print("\nUnidades encontradas:")
print(df_clean["duration_unit"].value_counts(dropna=False))


In [ ]:

# Estandarizar las unidades de duración
df_clean["duration_unit"] = (
    df_clean["duration_unit"]
    .replace({
        "Seasons": "Season"
    })
)

# Verificar el resultado
print("UNIDADES ESTANDARIZADAS")
print(
    df_clean["duration_unit"].value_counts(
        dropna=False
    )
)

# Validar que no haya valores sin duración
assert df_clean["duration_value"].notna().all(), \
    "Existen duraciones sin valor numérico"

print("\nValidación completada correctamente")


In [ ]:

# Seleccionar las columnas de la tabla principal
contenido = df_clean[
    [
        "show_id",
        "type",
        "title",
        "release_year",
        "rating",
        "duration_value",
        "duration_unit"
    ]
].copy()

# Validar la clave primaria
assert contenido["show_id"].notna().all()
assert contenido["show_id"].is_unique

# Verificar los resultados
print("TABLA CONTENIDO")
print("Registros:", len(contenido))
print("Columnas:", contenido.shape[1])

display(contenido.head())


In [ ]:

# Crear tabla de relación contenido-país
contenido_pais = df_clean[
    ["show_id", "country"]
].copy()

# Completar países faltantes
contenido_pais["country"] = (
    contenido_pais["country"]
    .fillna("Sin información")
)

# Separar países en filas independientes
contenido_pais["country"] = (
    contenido_pais["country"].str.split(",")
)

contenido_pais = contenido_pais.explode("country")

# Eliminar espacios sobrantes
contenido_pais["country"] = (
    contenido_pais["country"].str.strip()
)

# Eliminar relaciones duplicadas
contenido_pais = contenido_pais.drop_duplicates(
    subset=["show_id", "country"]
).reset_index(drop=True)

# Validaciones
assert contenido_pais["show_id"].isin(
    contenido["show_id"]
).all()

assert contenido_pais["country"].notna().all()

print("TABLA CONTENIDO_PAIS")
print("Registros:", len(contenido_pais))
print("Columnas:", contenido_pais.shape[1])

display(contenido_pais.head(10))


In [ ]:

# Crear tabla de relación contenido-género
contenido_genero = df_clean[
    ["show_id", "listed_in"]
].copy()

# Separar géneros en filas independientes
contenido_genero["listed_in"] = (
    contenido_genero["listed_in"].str.split(",")
)

contenido_genero = contenido_genero.explode("listed_in")

# Limpiar espacios y renombrar la columna
contenido_genero["listed_in"] = (
    contenido_genero["listed_in"].str.strip()
)

contenido_genero = contenido_genero.rename(
    columns={"listed_in": "genre"}
)

# Eliminar relaciones duplicadas
contenido_genero = contenido_genero.drop_duplicates(
    subset=["show_id", "genre"]
).reset_index(drop=True)

# Validar integridad referencial
assert contenido_genero["show_id"].isin(
    contenido["show_id"]
).all()

# Verificar resultados
print("TABLA CONTENIDO_GENERO")
print("Registros:", len(contenido_genero))
print("Columnas:", contenido_genero.shape[1])

display(contenido_genero.head(10))


In [ ]:

# Identificadores excluidos en la versión anterior
ids_revisar = ["s5965", "s5966", "s5967"]

# Revisar sus relaciones con países
paises_adicionales = contenido_pais[
    contenido_pais["show_id"].isin(ids_revisar)
]

# Revisar sus relaciones con géneros
generos_adicionales = contenido_genero[
    contenido_genero["show_id"].isin(ids_revisar)
]

print("RELACIONES DE PAÍSES")
display(paises_adicionales)
print("Cantidad:", len(paises_adicionales))

print("\nRELACIONES DE GÉNEROS")
display(generos_adicionales)
print("Cantidad:", len(generos_adicionales))


In [ ]:

# Identificadores ausentes en la versión anterior
ids_excluidos = ["s5965", "s5966", "s5967"]

# Relaciones de países sin esos tres contenidos
paises_comparables = contenido_pais[
    ~contenido_pais["show_id"].isin(ids_excluidos)
].copy()

print("RELACIONES DE PAÍSES COMPARABLES")
print("Pipeline actual:", len(paises_comparables))
print("MySQL anterior:", 10840)

print(
    "Diferencia:",
    len(paises_comparables) - 10840
)

# Comprobar duplicados
print(
    "Duplicados:",
    paises_comparables.duplicated(
        subset=["show_id", "country"]
    ).sum()
)


In [ ]:

# Leer la tabla de países del Excel anterior
archivo_excel = "netflix_dataset_final_normalizado.xlsx"

excel_paises = pd.read_excel(
    archivo_excel,
    sheet_name="contenido_pais"
)

# Estandarizar nombres y espacios
excel_paises.columns = excel_paises.columns.str.strip()

excel_paises["show_id"] = (
    excel_paises["show_id"].astype("string").str.strip()
)

excel_paises["country"] = (
    excel_paises["country"].astype("string").str.strip()
)

# Comparar las relaciones
comparacion = paises_comparables.merge(
    excel_paises[["show_id", "country"]].drop_duplicates(),
    on=["show_id", "country"],
    how="outer",
    indicator=True
)

print("RESULTADO DE LA COMPARACIÓN")
print(comparacion["_merge"].value_counts())

print("\nRELACIONES SOLO EN EL PIPELINE NUEVO")
display(
    comparacion.loc[
        comparacion["_merge"] == "left_only",
        ["show_id", "country"]
    ]
)

print("\nRELACIONES SOLO EN EL EXCEL ANTERIOR")
display(
    comparacion.loc[
        comparacion["_merge"] == "right_only",
        ["show_id", "country"]
    ]
)


In [ ]:

# Contar relaciones con países vacíos
vacios = (
    contenido_pais["country"]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
)

print("Relaciones vacías encontradas:", vacios.sum())

# Eliminar únicamente relaciones vacías
contenido_pais = contenido_pais.loc[~vacios].copy()

# Restablecer índices
contenido_pais = contenido_pais.reset_index(drop=True)

# Validar que no queden países vacíos
assert (
    contenido_pais["country"]
    .astype("string")
    .str.strip()
    .ne("")
    .all()
)

print("Relaciones después de limpiar:", len(contenido_pais))
print("Validación completada")


In [ ]:

# Validar tabla contenido
assert contenido["show_id"].is_unique
assert contenido["show_id"].notna().all()

# Validar tabla contenido_pais
assert not contenido_pais.duplicated(
    subset=["show_id", "country"]
).any()

assert contenido_pais["show_id"].isin(
    contenido["show_id"]
).all()

assert contenido_pais["country"].notna().all()

# Validar tabla contenido_genero
assert not contenido_genero.duplicated(
    subset=["show_id", "genre"]
).any()

assert contenido_genero["show_id"].isin(
    contenido["show_id"]
).all()

assert contenido_genero["genre"].notna().all()

# Resumen final
print("VALIDACIÓN FINAL DEL PIPELINE")
print("----------------------------")
print("Contenido:", len(contenido))
print("Países:", len(contenido_pais))
print("Géneros:", len(contenido_genero))
print("\nTodas las validaciones fueron exitosas")


In [ ]:

%pip install sqlalchemy pymysql


In [ ]:

import sqlalchemy
import pymysql

print("SQLAlchemy:", sqlalchemy.__version__)
print("PyMySQL instalado correctamente")
print("Entorno listo para conectar con MySQL")


In [ ]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

# Solicitar contraseña del usuario de ETL
clave = getpass("Contraseña de netflix_etl: ")

# Configurar la conexión
url_etl = URL.create(
    drivername="mysql+pymysql",
    username="netflix_etl",
    password=clave,
    host="127.0.0.1",
    port=3306,
    database="netflix_etl_dev"
)

engine_etl = create_engine(
    url_etl,
    pool_pre_ping=True
)

# Verificar conexión
with engine_etl.connect() as conexion:
    resultado = conexion.execute(
        text("SELECT CURRENT_USER(), DATABASE()")
    ).fetchone()

    print("CONEXIÓN EXITOSA")
    print("Usuario:", resultado[0])
    print("Base de datos:", resultado[1])

del clave


In [ ]:

from sqlalchemy import text

# Crear tablas en la base de pruebas
sql_tablas = [
    """
    CREATE TABLE IF NOT EXISTS contenido (
        show_id VARCHAR(10) PRIMARY KEY,
        type VARCHAR(20) NOT NULL,
        title VARCHAR(500) NOT NULL,
        release_year SMALLINT,
        rating VARCHAR(30),
        duration_value SMALLINT,
        duration_unit VARCHAR(20)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS contenido_pais (
        show_id VARCHAR(10) NOT NULL,
        country VARCHAR(150) NOT NULL,
        PRIMARY KEY (show_id, country),
        FOREIGN KEY (show_id)
            REFERENCES contenido(show_id)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS contenido_genero (
        show_id VARCHAR(10) NOT NULL,
        genre VARCHAR(150) NOT NULL,
        PRIMARY KEY (show_id, genre),
        FOREIGN KEY (show_id)
            REFERENCES contenido(show_id)
    )
    """
]

with engine_etl.begin() as conexion:
    for consulta in sql_tablas:
        conexion.execute(text(consulta))

print("Las tres tablas fueron creadas correctamente")


In [ ]:

from sqlalchemy import text

with engine_etl.connect() as conexion:
    for tabla in [
        "contenido",
        "contenido_pais",
        "contenido_genero"
    ]:
        cantidad = conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar()

        print(f"{tabla}: {cantidad} registros")


In [ ]:

from sqlalchemy import text

# Preparar los datos
datos_contenido = contenido[
    [
        "show_id", "type", "title",
        "release_year", "rating",
        "duration_value", "duration_unit"
    ]
].copy()

datos_pais = contenido_pais[
    ["show_id", "country"]
].copy()

datos_genero = contenido_genero[
    ["show_id", "genre"]
].copy()

# Comprobar que las tablas estén vacías
tablas = ["contenido", "contenido_pais", "contenido_genero"]

with engine_etl.connect() as conexion:
    cantidades = {
        tabla: conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar()
        for tabla in tablas
    }

if any(cantidad != 0 for cantidad in cantidades.values()):
    print("Carga cancelada: alguna tabla ya contiene registros.")
else:
    # Cargar los datos en una transacción
    with engine_etl.begin() as conexion:
        datos_contenido.to_sql(
            "contenido",
            con=conexion,
            if_exists="append",
            index=False,
            chunksize=1000
        )

        datos_pais.to_sql(
            "contenido_pais",
            con=conexion,
            if_exists="append",
            index=False,
            chunksize=1000
        )

        datos_genero.to_sql(
            "contenido_genero",
            con=conexion,
            if_exists="append",
            index=False,
            chunksize=1000
        )

    print("CARGA ETL COMPLETADA CORRECTAMENTE")


In [ ]:

from sqlalchemy import text

print("VALIDACIÓN DE CARGA ETL")
print("-------------------------")

tablas = {
    "contenido": len(contenido),
    "contenido_pais": len(contenido_pais),
    "contenido_genero": len(contenido_genero)
}

with engine_etl.connect() as conexion:
    for tabla, esperado in tablas.items():
        obtenido = conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar()

        estado = "OK" if obtenido == esperado else "ERROR"

        print(
            f"{tabla}: "
            f"Esperados={esperado} | "
            f"MySQL={obtenido} | "
            f"Estado={estado}"
        )


In [ ]:

from sqlalchemy import text
import pandas as pd

consultas = {
    "Paises sin contenido": """
        SELECT COUNT(*)
        FROM contenido_pais p
        LEFT JOIN contenido c
            ON p.show_id = c.show_id
        WHERE c.show_id IS NULL
    """,
    "Generos sin contenido": """
        SELECT COUNT(*)
        FROM contenido_genero g
        LEFT JOIN contenido c
            ON g.show_id = c.show_id
        WHERE c.show_id IS NULL
    """,
    "IDs duplicados en contenido": """
        SELECT COUNT(*) - COUNT(DISTINCT show_id)
        FROM contenido
    """,
    "Relaciones de pais duplicadas": """
        SELECT COUNT(*) - COUNT(DISTINCT show_id, country)
        FROM contenido_pais
    """,
    "Relaciones de genero duplicadas": """
        SELECT COUNT(*) - COUNT(DISTINCT show_id, genre)
        FROM contenido_genero
    """
}

resultados = []

with engine_etl.connect() as conexion:
    for validacion, consulta in consultas.items():
        errores = conexion.execute(
            text(consulta)
        ).scalar()

        resultados.append({
            "Validacion": validacion,
            "Errores": errores,
            "Estado": "OK" if errores == 0 else "REVISAR"
        })

reporte = pd.DataFrame(resultados)
display(reporte)


In [ ]:

import pandas as pd
from sqlalchemy import text

consulta = """
SELECT
    type AS tipo_contenido,
    COUNT(*) AS cantidad
FROM contenido
GROUP BY type
ORDER BY cantidad DESC
"""

with engine_etl.connect() as conexion:
    resumen = pd.read_sql(
        text(consulta),
        con=conexion
    )

display(resumen)


In [ ]:

# Verificar los componentes necesarios
componentes = {
    "Datos originales": "df_raw",
    "Tabla contenido": "contenido",
    "Tabla países": "contenido_pais",
    "Tabla géneros": "contenido_genero",
    "Conexión MySQL": "engine_etl"
}

for nombre, variable in componentes.items():
    estado = (
        "DISPONIBLE"
        if variable in globals()
        else "NO DISPONIBLE"
    )
    print(f"{nombre}: {estado}")


In [ ]:

def validar_datos(contenido, contenido_pais, contenido_genero):

    # Validar identificadores únicos
    assert contenido["show_id"].is_unique, (
        "Existen contenidos duplicados"
    )

    # Validar identificadores no nulos
    assert contenido["show_id"].notna().all(), (
        "Existen identificadores vacíos"
    )

    # Validar relaciones de países
    assert not contenido_pais.duplicated(
        subset=["show_id", "country"]
    ).any(), "Existen relaciones de países duplicadas"

    # Validar relaciones de géneros
    assert not contenido_genero.duplicated(
        subset=["show_id", "genre"]
    ).any(), "Existen relaciones de géneros duplicadas"

    # Validar integridad referencial
    assert contenido_pais["show_id"].isin(
        contenido["show_id"]
    ).all(), "Existen países sin contenido"

    assert contenido_genero["show_id"].isin(
        contenido["show_id"]
    ).all(), "Existen géneros sin contenido"

    # Validar valores obligatorios
    assert contenido_pais["country"].notna().all()
    assert contenido_genero["genre"].notna().all()

    assert contenido_pais["country"].astype(
        "string"
    ).str.strip().ne("").all()

    assert contenido_genero["genre"].astype(
        "string"
    ).str.strip().ne("").all()

    print("VALIDACIÓN AUTOMÁTICA EXITOSA")
    print("Contenidos:", len(contenido))
    print("Relaciones de países:", len(contenido_pais))
    print("Relaciones de géneros:", len(contenido_genero))


In [ ]:

validar_datos(
    contenido,
    contenido_pais,
    contenido_genero
)


In [ ]:

from pathlib import Path
import pandas as pd

def extraer_datos(ruta_archivo):
    """
    Lee el archivo CSV original de Netflix
    y devuelve un DataFrame sin modificarlo.
    """

    ruta = Path(ruta_archivo)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}"
        )

    # Leer el archivo original
    datos = pd.read_csv(
        ruta,
        encoding="latin-1"
    )

    print("EXTRACCIÓN COMPLETADA")
    print("Filas:", datos.shape[0])
    print("Columnas:", datos.shape[1])

    return datos


In [ ]:

# Ejecutar extracción automática
datos_extraidos = extraer_datos(
    "netflix_titles.csv"
)

# Mostrar los primeros registros
display(datos_extraidos.head())


In [ ]:

def limpiar_datos_base(datos):
    """
    Realiza la limpieza inicial del dataset Netflix.
    """

    df = datos.copy()

    # Eliminar columnas sin información
    df = df.dropna(axis=1, how="all")

    # Detectar duraciones ubicadas incorrectamente
    patron = r"^\s*\d+\s*(?:min|Seasons?)\s*$"

    mascara = (
        df["duration"].isna()
        & df["rating"].astype("string").str.fullmatch(
            patron, case=False
        ).fillna(False)
    )

    # Corregir los registros identificados
    cantidad_corregida = int(mascara.sum())

    df.loc[mascara, "duration"] = df.loc[
        mascara, "rating"
    ]

    df.loc[mascara, "rating"] = None

    print("LIMPIEZA INICIAL COMPLETADA")
    print("Filas:", len(df))
    print("Columnas:", len(df.columns))
    print("Duraciones corregidas:", cantidad_corregida)

    return df


In [ ]:

datos_limpios = limpiar_datos_base(
    datos_extraidos
)


In [ ]:

def normalizar_duraciones(datos):
    """
    Separa la duración en valor numérico y unidad.
    """

    df = datos.copy()

    # Extraer la cantidad numérica
    df["duration_value"] = pd.to_numeric(
        df["duration"].astype("string").str.extract(
            r"^(\d+)", expand=False
        ),
        errors="coerce"
    )

    # Estandarizar la unidad de duración
    df["duration_unit"] = (
        df["duration"]
        .astype("string")
        .str.extract(
            r"^\d+\s*(min|Seasons?)$",
            expand=False
        )
        .replace({
            "Seasons": "Season"
        })
    )

    print("NORMALIZACIÓN DE DURACIONES")
    print(
        df.groupby(
            ["type", "duration_unit"]
        ).size()
    )

    print(
        "Duraciones sin valor:",
        df["duration_value"].isna().sum()
    )

    return df


In [ ]:

datos_normalizados = normalizar_duraciones(
    datos_limpios
)


In [ ]:

def crear_tablas_relacionales(datos):
    """
    Construye las tres tablas del modelo relacional.
    """

    # Tabla principal
    contenido_nuevo = datos[
        [
            "show_id",
            "type",
            "title",
            "release_year",
            "rating",
            "duration_value",
            "duration_unit"
        ]
    ].copy()

    # Tabla de países
    contenido_pais_nuevo = datos[
        ["show_id", "country"]
    ].copy()

    contenido_pais_nuevo["country"] = (
        contenido_pais_nuevo["country"]
        .fillna("Sin información")
        .str.split(",")
    )

    contenido_pais_nuevo = (
        contenido_pais_nuevo
        .explode("country")
    )

    contenido_pais_nuevo["country"] = (
        contenido_pais_nuevo["country"]
        .str.strip()
    )

    contenido_pais_nuevo = (
        contenido_pais_nuevo[
            contenido_pais_nuevo["country"].notna()
            & contenido_pais_nuevo["country"].ne("")
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Tabla de géneros
    contenido_genero_nuevo = datos[
        ["show_id", "listed_in"]
    ].copy()

    contenido_genero_nuevo["listed_in"] = (
        contenido_genero_nuevo["listed_in"]
        .str.split(",")
    )

    contenido_genero_nuevo = (
        contenido_genero_nuevo
        .explode("listed_in")
    )

    contenido_genero_nuevo = (
        contenido_genero_nuevo
        .rename(columns={"listed_in": "genre"})
    )

    contenido_genero_nuevo["genre"] = (
        contenido_genero_nuevo["genre"]
        .str.strip()
    )

    contenido_genero_nuevo = (
        contenido_genero_nuevo[
            contenido_genero_nuevo["genre"].notna()
            & contenido_genero_nuevo["genre"].ne("")
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print("TABLAS RELACIONALES CREADAS")
    print("Contenido:", len(contenido_nuevo))
    print("Países:", len(contenido_pais_nuevo))
    print("Géneros:", len(contenido_genero_nuevo))

    return (
        contenido_nuevo,
        contenido_pais_nuevo,
        contenido_genero_nuevo
    )


In [ ]:

(
    contenido_nuevo,
    contenido_pais_nuevo,
    contenido_genero_nuevo
) = crear_tablas_relacionales(
    datos_normalizados
)


In [ ]:

def ejecutar_pipeline_transformacion(ruta_archivo):
    """
    Ejecuta la extracción, limpieza,
    transformación y validación de Netflix.
    """

    print("INICIANDO PIPELINE ETL")
    print("----------------------")

    # 1. Extracción
    datos = extraer_datos(ruta_archivo)

    # 2. Limpieza inicial
    datos = limpiar_datos_base(datos)

    # 3. Normalización de duraciones
    datos = normalizar_duraciones(datos)

    # 4. Creación de tablas relacionales
    contenido_final, pais_final, genero_final = (
        crear_tablas_relacionales(datos)
    )

    # 5. Validación automática
    validar_datos(
        contenido_final,
        pais_final,
        genero_final
    )

    print("----------------------")
    print("PIPELINE DE TRANSFORMACIÓN COMPLETADO")

    return contenido_final, pais_final, genero_final


In [ ]:

(
    contenido_etl,
    pais_etl,
    genero_etl
) = ejecutar_pipeline_transformacion(
    "netflix_titles.csv"
)


In [ ]:

from sqlalchemy import text

def verificar_destino_carga(engine, contenido, pais, genero):
    """
    Comprueba la base de datos de destino
    y los registros existentes antes de cargar.
    """

    esperados = {
        "contenido": len(contenido),
        "contenido_pais": len(pais),
        "contenido_genero": len(genero)
    }

    with engine.connect() as conexion:

        base_actual = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar()

        if base_actual != "netflix_etl_dev":
            raise ValueError(
                f"Base de datos no autorizada: {base_actual}"
            )

        print("BASE DE DATOS:", base_actual)
        print("----------------------------")

        for tabla, esperado in esperados.items():

            actual = conexion.execute(
                text(f"SELECT COUNT(*) FROM {tabla}")
            ).scalar()

            print(
                f"{tabla}: "
                f"MySQL={actual} | "
                f"Pipeline={esperado}"
            )

    print("----------------------------")
    print("VERIFICACIÓN DE DESTINO COMPLETADA")


In [ ]:

verificar_destino_carga(
    engine_etl,
    contenido_etl,
    pais_etl,
    genero_etl
)


In [ ]:

import pandas as pd
from sqlalchemy import text

def comparar_con_mysql(engine, tablas_pipeline):

    resultados = []

    with engine.connect() as conexion:

        base = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar()

        if base != "netflix_etl_dev":
            raise ValueError(
                f"Base de datos no autorizada: {base}"
            )

        for nombre, datos_pipeline in tablas_pipeline.items():

            datos_mysql = pd.read_sql(
                text(f"SELECT * FROM {nombre}"),
                conexion
            )

            columnas = list(datos_pipeline.columns)

            # Comparar los datos sin depender del orden
            origen = datos_pipeline[columnas].copy()
            destino = datos_mysql[columnas].copy()

            for columna in columnas:
                if columna == "duration_value":
                    origen[columna] = pd.to_numeric(
                        origen[columna], errors="coerce"
                    )
                    destino[columna] = pd.to_numeric(
                        destino[columna], errors="coerce"
                    )
                else:
                    origen[columna] = origen[columna].astype("string")
                    destino[columna] = destino[columna].astype("string")

            claves = {
                "contenido": ["show_id"],
                "contenido_pais": ["show_id", "country"],
                "contenido_genero": ["show_id", "genre"]
            }[nombre]

            origen = origen.sort_values(
                claves
            ).reset_index(drop=True)

            destino = destino.sort_values(
                claves
            ).reset_index(drop=True)

            iguales = origen.equals(destino)

            resultados.append({
                "Tabla": nombre,
                "Registros": len(origen),
                "Coinciden": "SI" if iguales else "NO"
            })

    return pd.DataFrame(resultados)


In [ ]:

tablas_pipeline = {
    "contenido": contenido_etl,
    "contenido_pais": pais_etl,
    "contenido_genero": genero_etl
}

resultado_comparacion = comparar_con_mysql(
    engine_etl,
    tablas_pipeline
)

display(resultado_comparacion)


In [ ]:

# Consultar la tabla contenido desde MySQL
with engine_etl.connect() as conexion:
    contenido_mysql = pd.read_sql(
        text("SELECT * FROM contenido"),
        conexion
    )

# Preparar los dos DataFrames
origen = contenido_etl.copy()
destino = contenido_mysql.copy()

# Ordenar por identificador
origen = origen.sort_values(
    "show_id"
).reset_index(drop=True)

destino = destino.sort_values(
    "show_id"
).reset_index(drop=True)

# Comparar columna por columna
print("COMPARACIÓN POR COLUMNAS")
print("-----------------------")

for columna in origen.columns:

    valores_origen = origen[columna].astype("string")
    valores_destino = destino[columna].astype("string")

    diferencias = (
        valores_origen.fillna("<NULO>")
        != valores_destino.fillna("<NULO>")
    )

    print(
        f"{columna}: {diferencias.sum()} diferencias"
    )


In [ ]:

from pandas.testing import assert_frame_equal

# Preparar los datos
origen_final = origen.copy()
destino_final = destino.copy()

# Normalizar valores para la comparación
for columna in origen_final.columns:
    origen_final[columna] = (
        origen_final[columna]
        .astype("string")
        .fillna("<NULO>")
    )

    destino_final[columna] = (
        destino_final[columna]
        .astype("string")
        .fillna("<NULO>")
    )

# Verificar igualdad
try:
    assert_frame_equal(
        origen_final,
        destino_final,
        check_dtype=False
    )

    print("COMPARACIÓN EXITOSA")
    print("Los 8809 registros coinciden con MySQL.")

except AssertionError as error:
    print("SE DETECTARON DIFERENCIAS")
    print(error)


In [ ]:

from sqlalchemy import text

def cargar_datos(engine, contenido, pais, genero):
    """
    Carga los datos en MySQL únicamente
    cuando las tres tablas están vacías.
    """

    tablas = {
        "contenido": contenido,
        "contenido_pais": pais,
        "contenido_genero": genero
    }

    # Validar los datos antes de cargar
    validar_datos(contenido, pais, genero)

    with engine.begin() as conexion:

        # Verificar la base de datos
        base_actual = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar()

        if base_actual != "netflix_etl_dev":
            raise ValueError(
                f"Base de datos no autorizada: {base_actual}"
            )

        # Comprobar si ya existen registros
        cantidades = {}

        for tabla in tablas:
            cantidades[tabla] = conexion.execute(
                text(f"SELECT COUNT(*) FROM {tabla}")
            ).scalar()

        if any(cantidad > 0 for cantidad in cantidades.values()):
            print("CARGA OMITIDA")
            print("Las tablas ya contienen registros.")
            print("No se modificó la base de datos.")
            return

        # Insertar respetando las relaciones
        for nombre, datos in tablas.items():
            datos.to_sql(
                name=nombre,
                con=conexion,
                if_exists="append",
                index=False,
                chunksize=1000
            )

        print("CARGA ETL COMPLETADA")
        print("Registros insertados:")
        for nombre, datos in tablas.items():
            print(f"{nombre}: {len(datos)}")


In [ ]:

cargar_datos(
    engine_etl,
    contenido_etl,
    pais_etl,
    genero_etl
)


In [ ]:

def ejecutar_pipeline_etl(ruta_archivo, engine):
    """
    Ejecuta el pipeline completo de Netflix:
    extracción, transformación, validación
    y carga a MySQL.
    """

    print("================================")
    print("INICIANDO PIPELINE ETL NETFLIX")
    print("================================")

    # 1. Extraer, transformar y validar
    contenido, pais, genero = (
        ejecutar_pipeline_transformacion(
            ruta_archivo
        )
    )

    # 2. Cargar en MySQL
    cargar_datos(
        engine,
        contenido,
        pais,
        genero
    )

    print("================================")
    print("EJECUCIÓN DEL PIPELINE FINALIZADA")
    print("================================")


In [ ]:

ejecutar_pipeline_etl(
    "netflix_titles.csv",
    engine_etl
)


In [ ]:

import sys
import pandas as pd
import sqlalchemy
import pymysql

print("TECNOLOGÍAS DEL PROYECTO")
print("------------------------")

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("PyMySQL:", pymysql.__version__)

with engine_etl.connect() as conexion:
    version_mysql = conexion.execute(
        text("SELECT VERSION()")
    ).scalar()

print("MySQL:", version_mysql)


In [ ]:

from pathlib import Path

dependencias = [
    f"pandas=={pd.__version__}",
    f"SQLAlchemy=={sqlalchemy.__version__}",
    f"PyMySQL=={pymysql.__version__}"
]

ruta_requirements = Path("requirements.txt")

ruta_requirements.write_text(
    "\n".join(dependencias) + "\n",
    encoding="utf-8"
)

print("ARCHIVO REQUIREMENTS CREADO")
print("---------------------------")
print(ruta_requirements.resolve())
print()
print(ruta_requirements.read_text(encoding="utf-8"))


In [ ]:

from pathlib import Path
import shutil

# Carpeta principal del proyecto
proyecto = Path.home() / "netflix-etl-pipeline"

# Crear las carpetas necesarias
carpetas = [
    "data",
    "notebooks",
    "src",
    "sql"
]

proyecto.mkdir(exist_ok=True)

for carpeta in carpetas:
    (proyecto / carpeta).mkdir(exist_ok=True)

# Copiar requirements.txt a la carpeta del proyecto
origen = Path("requirements.txt")
destino = proyecto / "requirements.txt"

shutil.copy2(origen, destino)

print("ESTRUCTURA DEL PROYECTO CREADA")
print("-----------------------------")
print("Ubicación:", proyecto)
print()

for elemento in sorted(proyecto.iterdir()):
    print(
        "[CARPETA]" if elemento.is_dir() else "[ARCHIVO]",
        elemento.name
    )


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

contenido_gitignore = """
# Credenciales y variables de entorno
.env
.env.*
!.env.example
*.pem
*.key

# Archivos temporales de Python
__pycache__/
*.pyc

# Entornos virtuales
.venv/
venv/

# Archivos temporales de Jupyter
.ipynb_checkpoints/

# Configuración del editor
.vscode/
.idea/

# Archivos de sistema
.DS_Store
Thumbs.db
"""

ruta_gitignore = proyecto / ".gitignore"

ruta_gitignore.write_text(
    contenido_gitignore.strip() + "\n",
    encoding="utf-8"
)

print("ARCHIVO .gitignore CREADO")
print("------------------------")
print("Ubicación:", ruta_gitignore)
print()
print(ruta_gitignore.read_text(encoding="utf-8"))


In [ ]:

from pathlib import Path
import shutil

# Ubicación del proyecto
proyecto = Path.home() / "netflix-etl-pipeline"

# Archivo original
origen_csv = Path("netflix_titles.csv")

# Destino dentro del proyecto
destino_csv = proyecto / "data" / "netflix_titles.csv"

# Comprobar que existe el archivo original
if not origen_csv.is_file():
    print("ERROR: No se encontró el archivo CSV original.")
else:
    # Copiar sin modificar el original
    shutil.copy2(origen_csv, destino_csv)

    print("ARCHIVO CSV COPIADO CORRECTAMENTE")
    print("--------------------------------")
    print("Ubicación:", destino_csv)
    print("Tamaño:", destino_csv.stat().st_size, "bytes")


In [ ]:

from pathlib import Path

# Buscar notebooks en la carpeta actual
carpeta_actual = Path.cwd()

print("CARPETA ACTUAL:")
print(carpeta_actual)

print("\nNOTEBOOKS ENCONTRADOS:")
print("---------------------")

notebooks = list(
    carpeta_actual.glob("*.ipynb")
)

if notebooks:
    for archivo in notebooks:
        print(archivo.name)
else:
    print("No se encontraron notebooks en esta carpeta.")


In [ ]:

from pathlib import Path
import json

# Rutas
origen = Path.home() / "01_netflix_pipeline_etl.ipynb"

proyecto = Path.home() / "netflix-etl-pipeline"
destino = proyecto / "notebooks" / "01_netflix_pipeline_etl.ipynb"

# Leer el notebook original
with origen.open("r", encoding="utf-8") as archivo:
    notebook = json.load(archivo)

# Limpiar resultados de ejecución de la copia
for celda in notebook["cells"]:
    if celda["cell_type"] == "code":
        celda["outputs"] = []
        celda["execution_count"] = None

# Guardar la copia
with destino.open("w", encoding="utf-8") as archivo:
    json.dump(
        notebook,
        archivo,
        ensure_ascii=False,
        indent=1
    )

print("COPIA DEL NOTEBOOK CREADA")
print("------------------------")
print("Ubicación:", destino)
print("Celdas:", len(notebook["cells"]))
print("Salidas de ejecución eliminadas")
print("Notebook original conservado")


In [ ]:
from pathlib import Path
import json
import re

In [ ]:

from pathlib import Path
import json
import re

ruta = (
    Path.home()
    / "netflix-etl-pipeline"
    / "notebooks"
    / "01_netflix_pipeline_etl.ipynb"
)

with open(ruta, "r", encoding="utf-8") as archivo:
    notebook = json.load(archivo)

patrones = {
    "Posible contraseña": r"(?i)(password|passwd|contraseña|clave)\s*=",
    "Contraseña SQL": r"(?i)IDENTIFIED\s+BY",
    "Conexión MySQL": r"(?i)mysql\+pymysql://",
    "Archivo .env": r"(?i)\.env"
}

print("REVISIÓN DE SEGURIDAD")
print("---------------------")

hallazgos = 0

for numero, celda in enumerate(notebook["cells"], 1):
    codigo = "".join(celda.get("source", []))

    for descripcion, patron in patrones.items():
        if re.search(patron, codigo):
            print(f"Celda {numero}: {descripcion}")
            hallazgos += 1

print("---------------------")
print("Coincidencias encontradas:", hallazgos)
print("Revisión finalizada")


In [ ]:

import re

print("REVISIÓN DE CELDAS")
print("------------------")

for numero in [22, 55]:

    celda = notebook["cells"][numero - 1]
    codigo = "".join(celda.get("source", []))

    print(f"\nCelda {numero}")

    print(
        "Utiliza getpass:",
        bool(re.search(r"\bgetpass\s*\(", codigo))
    )

    print(
        "Contiene asignación de contraseña:",
        bool(re.search(
            r"(?i)(password|passwd|contraseña|clave)\s*=",
            codigo
        ))
    )

    print(
        "Contiene una referencia a .env:",
        ".env" in codigo
    )

    print(
        "Contiene instrucción SQL de contraseña:",
        bool(re.search(r"(?i)IDENTIFIED\s+BY", codigo))
    )


In [ ]:

import ast

codigo = "".join(
    notebook["cells"][21].get("source", [])
)

print("ANÁLISIS DE SEGURIDAD - CELDA 22")
print("--------------------------------")

try:
    arbol = ast.parse(codigo)
    asignaciones = []

    for nodo in ast.walk(arbol):
        if isinstance(nodo, (ast.Assign, ast.AnnAssign)):
            valor = nodo.value

            if isinstance(valor, ast.Call):
                funcion = valor.func

                if isinstance(funcion, ast.Name):
                    asignaciones.append(funcion.id)

                elif isinstance(funcion, ast.Attribute):
                    asignaciones.append(funcion.attr)

            elif isinstance(valor, ast.Constant):
                if isinstance(valor.value, str):
                    asignaciones.append(
                        "VALOR DE TEXTO DIRECTO"
                    )

    print("Tipos de asignación encontrados:")

    for tipo in asignaciones:
        print("-", tipo)

except SyntaxError:
    print("No fue posible analizar esta celda.")


In [ ]:

import json
import re
from pathlib import Path

ruta = (
    Path.home()
    / "netflix-etl-pipeline"
    / "notebooks"
    / "01_netflix_pipeline_etl.ipynb"
)

with ruta.open("r", encoding="utf-8") as archivo:
    nb = json.load(archivo)

print("AUDITORÍA DEL NOTEBOOK")
print("----------------------")

salidas = 0
posibles_secretos = []

patrones = {
    "Contraseña literal": r"""(?i)\b(?:password|passwd|contraseña|clave)\s*=\s*['"][^'"]+['"]""",
    "Contraseña SQL": r"(?i)IDENTIFIED\s+BY\s+['\"]",
    "URL con credenciales": r"(?i)mysql\+pymysql://[^@\s]+@"
}

for numero, celda in enumerate(nb["cells"], 1):

    if celda["cell_type"] != "code":
        continue

    salidas += len(celda.get("outputs", []))

    codigo = "".join(celda.get("source", []))

    for descripcion, patron in patrones.items():
        if re.search(patron, codigo):
            posibles_secretos.append(
                (numero, descripcion)
            )

print("Salidas guardadas:", salidas)
print("Posibles secretos:", len(posibles_secretos))

for numero, descripcion in posibles_secretos:
    print(f"Celda {numero}: {descripcion}")

print("----------------------")
print("AUDITORÍA FINALIZADA")


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

codigo_extract = '''from pathlib import Path
import pandas as pd


def extraer_datos(ruta_archivo):
    """
    Extrae los datos originales de Netflix.
    """

    ruta = Path(ruta_archivo)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}"
        )

    datos = pd.read_csv(
        ruta,
        encoding="latin-1"
    )

    print("EXTRACCIÓN COMPLETADA")
    print("Filas:", len(datos))
    print("Columnas:", len(datos.columns))

    return datos
'''

ruta_extract = proyecto / "src" / "extract.py"

ruta_extract.write_text(
    codigo_extract,
    encoding="utf-8"
)

print("ARCHIVO PYTHON CREADO")
print("--------------------")
print("Ubicación:", ruta_extract)
print("Existe:", ruta_extract.is_file())


In [ ]:

import sys
from pathlib import Path

# Ubicación del proyecto
proyecto = Path.home() / "netflix-etl-pipeline"

# Agregar la carpeta src para importar el módulo
ruta_src = str(proyecto / "src")

if ruta_src not in sys.path:
    sys.path.insert(0, ruta_src)

# Importar la función desde extract.py
from extract import extraer_datos

# Leer el CSV desde la carpeta del proyecto
ruta_csv = proyecto / "data" / "netflix_titles.csv"

datos_prueba = extraer_datos(ruta_csv)

print("---------------------------")
print("PRUEBA DEL MÓDULO EXTRACT")
print("Estado:", "CORRECTO" if datos_prueba.shape == (8809, 26) else "REVISAR")


In [ ]:

import ast

funciones_buscadas = [
    "limpiar_datos_base",
    "normalizar_duraciones",
    "crear_tablas_relacionales"
]

print("FUNCIONES DE TRANSFORMACIÓN")
print("---------------------------")

encontradas = {}

for numero, celda in enumerate(notebook["cells"], 1):
    if celda["cell_type"] != "code":
        continue

    codigo = "".join(celda.get("source", []))

    try:
        arbol = ast.parse(codigo)
    except SyntaxError:
        continue

    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef):
            if nodo.name in funciones_buscadas:
                encontradas[nodo.name] = numero
                print(f"{nodo.name}: celda {numero}")

print("---------------------------")
print("Funciones encontradas:", len(encontradas))
print("Funciones esperadas:", len(funciones_buscadas))


In [ ]:

import ast
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

funciones_necesarias = [
    "limpiar_datos_base",
    "normalizar_duraciones",
    "crear_tablas_relacionales"
]

codigo_funciones = {}

# Extraer las funciones originales del notebook
for celda in notebook["cells"]:

    if celda["cell_type"] != "code":
        continue

    codigo = "".join(celda.get("source", []))

    try:
        arbol = ast.parse(codigo)
    except SyntaxError:
        continue

    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef):
            if nodo.name in funciones_necesarias:
                codigo_funciones[nodo.name] = ast.get_source_segment(
                    codigo, nodo
                )

# Verificar que estén las tres funciones
faltantes = [
    nombre for nombre in funciones_necesarias
    if nombre not in codigo_funciones
]

if faltantes:
    print("Faltan funciones:", faltantes)

else:
    encabezado = "import pandas as pd\n\n\n"

    contenido = encabezado + "\n\n\n".join(
        codigo_funciones[nombre]
        for nombre in funciones_necesarias
    ) + "\n"

    ruta_transform = proyecto / "src" / "transform.py"

    ruta_transform.write_text(
        contenido,
        encoding="utf-8"
    )

    print("ARCHIVO TRANSFORM.PY CREADO")
    print("---------------------------")
    print("Existe:", ruta_transform.is_file())
    print("Funciones incluidas:", len(funciones_necesarias))

    # Comprobar que el archivo tiene sintaxis válida
    ast.parse(contenido)
    print("Sintaxis Python: CORRECTA")


In [ ]:

import importlib
import transform

# Recargar el módulo desde transform.py
importlib.reload(transform)

# Trabajar sobre una copia de los datos originales
datos_limpios = transform.limpiar_datos_base(
    datos_prueba.copy()
)

datos_normalizados = transform.normalizar_duraciones(
    datos_limpios
)

contenido_test, pais_test, genero_test = (
    transform.crear_tablas_relacionales(
        datos_normalizados
    )
)

print("\nPRUEBA DEL MÓDULO TRANSFORM")
print("---------------------------")
print("Contenido:", len(contenido_test))
print("Países:", len(pais_test))
print("Géneros:", len(genero_test))

resultado_correcto = (
    len(contenido_test) == 8809
    and len(pais_test) == 10845
    and len(genero_test) == 19329
)

print(
    "Estado:",
    "CORRECTO" if resultado_correcto else "REVISAR"
)


In [ ]:

import ast
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

nombre_funcion = "validar_datos"
codigo_validacion = None

# Buscar la función original en el notebook
for celda in notebook["cells"]:

    if celda["cell_type"] != "code":
        continue

    codigo = "".join(celda.get("source", []))

    try:
        arbol = ast.parse(codigo)
    except SyntaxError:
        continue

    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef):
            if nodo.name == nombre_funcion:
                codigo_validacion = ast.get_source_segment(
                    codigo, nodo
                )

if codigo_validacion is None:
    print("No se encontró la función validar_datos.")

else:
    contenido_archivo = (
        "import pandas as pd\n\n\n"
        + codigo_validacion
        + "\n"
    )

    # Comprobar sintaxis antes de guardar
    ast.parse(contenido_archivo)

    ruta_validate = proyecto / "src" / "validate.py"

    ruta_validate.write_text(
        contenido_archivo,
        encoding="utf-8"
    )

    print("ARCHIVO VALIDATE.PY CREADO")
    print("-------------------------")
    print("Existe:", ruta_validate.is_file())
    print("Función incluida:", nombre_funcion)
    print("Sintaxis Python: CORRECTA")


In [ ]:

import importlib
import validate

# Recargar el módulo desde validate.py
importlib.reload(validate)

print("PRUEBA DEL MÓDULO VALIDATE")
print("-------------------------")

# Validar las tres tablas transformadas
resultado = validate.validar_datos(
    contenido_test,
    pais_test,
    genero_test
)

print("-------------------------")
print("Validación ejecutada correctamente")
print("Resultado devuelto:", resultado)


In [ ]:

import ast
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

nombre_funcion = "cargar_datos"
codigo_carga = None

# Buscar la función original en el notebook
for celda in notebook["cells"]:

    if celda["cell_type"] != "code":
        continue

    codigo = "".join(celda.get("source", []))

    try:
        arbol = ast.parse(codigo)
    except SyntaxError:
        continue

    for nodo in arbol.body:
        if isinstance(nodo, ast.FunctionDef):
            if nodo.name == nombre_funcion:
                codigo_carga = ast.get_source_segment(
                    codigo, nodo
                )

if codigo_carga is None:
    print("No se encontró la función cargar_datos.")

else:
    contenido_archivo = (
        "import pandas as pd\n"
        "from sqlalchemy import text\n\n\n"
        + codigo_carga
        + "\n"
    )

    # Verificar sintaxis antes de guardar
    ast.parse(contenido_archivo)

    ruta_load = proyecto / "src" / "load.py"

    ruta_load.write_text(
        contenido_archivo,
        encoding="utf-8"
    )

    print("ARCHIVO LOAD.PY CREADO")
    print("---------------------")
    print("Existe:", ruta_load.is_file())
    print("Función incluida:", nombre_funcion)
    print("Sintaxis Python: CORRECTA")


In [ ]:


from pathlib import Path

ruta_load = (
    Path.home()
    / "netflix-etl-pipeline"
    / "src"
    / "load.py"
)

codigo = ruta_load.read_text(encoding="utf-8")

importacion = "from validate import validar_datos"

if importacion not in codigo:
    codigo = importacion + "\n" + codigo
    ruta_load.write_text(codigo, encoding="utf-8")

print("CORRECCIÓN DE LOAD.PY")
print("---------------------")
print("Importación agregada correctamente")
print("Archivo actualizado:", ruta_load.is_file())


In [ ]:

from pathlib import Path

ruta_load = (
    Path.home()
    / "netflix-etl-pipeline"
    / "src"
    / "load.py"
)

lineas = ruta_load.read_text(
    encoding="utf-8"
).splitlines()

print("REVISIÓN DE LOAD.PY")
print("------------------")

for numero in range(15, 23):
    if numero <= len(lineas):
        print(f"Línea {numero}: {lineas[numero - 1]}")


In [ ]:

import importlib
import load

# Actualizar el módulo con la corrección
importlib.reload(load)

print("SEGUNDA PRUEBA DEL MÓDULO LOAD")
print("-----------------------------")

load.cargar_datos(
    engine_etl,
    contenido_test,
    pais_test,
    genero_test
)

print("-----------------------------")
print("Prueba finalizada")


In [ ]:

from sqlalchemy import text

print("VERIFICACIÓN DE MYSQL")
print("---------------------")

with engine_etl.connect() as conexion:
    for tabla in [
        "contenido",
        "contenido_pais",
        "contenido_genero"
    ]:
        cantidad = conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla}")
        ).scalar_one()

        print(f"{tabla}: {cantidad} registros")

print("---------------------")
print("Verificación finalizada")


In [ ]:

import ast
from pathlib import Path

ruta_load = (
    Path.home()
    / "netflix-etl-pipeline"
    / "src"
    / "load.py"
)

codigo = ruta_load.read_text(encoding="utf-8")
arbol = ast.parse(codigo)

print("REVISIÓN DEL MÓDULO LOAD")
print("-----------------------")

print("Importaciones:")
for nodo in arbol.body:
    if isinstance(nodo, ast.Import):
        for modulo in nodo.names:
            print("-", modulo.name)

    elif isinstance(nodo, ast.ImportFrom):
        print("-", nodo.module)

print("\nFunciones definidas:")
for nodo in arbol.body:
    if isinstance(nodo, ast.FunctionDef):
        print("-", nodo.name)

print("\nEstructura Python: CORRECTA")


In [ ]:

import ast
from pathlib import Path

ruta_load = (
    Path.home()
    / "netflix-etl-pipeline"
    / "src"
    / "load.py"
)

codigo = ruta_load.read_text(encoding="utf-8")
lineas = codigo.splitlines()

print("REVISIÓN DE LA LÓGICA DE CARGA")
print("-----------------------------")

palabras_clave = [
    "SELECT COUNT",
    "CARGA OMITIDA",
    "to_sql",
    "return"
]

for numero, linea in enumerate(lineas, 1):
    if any(palabra in linea for palabra in palabras_clave):
        print(f"Línea {numero}: {linea.strip()}")

print("-----------------------------")
print("Revisión finalizada")


In [ ]:

from pathlib import Path

ruta_load = (
    Path.home()
    / "netflix-etl-pipeline"
    / "src"
    / "load.py"
)

lineas = ruta_load.read_text(
    encoding="utf-8"
).splitlines()

print("CONTROL DE DUPLICADOS")
print("---------------------")

for numero in range(25, 55):
    if numero <= len(lineas):
        print(f"{numero}: {lineas[numero - 1]}")


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

codigo_main = '''from pathlib import Path

from extract import extraer_datos
from transform import (
    limpiar_datos_base,
    normalizar_duraciones,
    crear_tablas_relacionales
)
from validate import validar_datos


def main():
    proyecto = Path(__file__).resolve().parent.parent
    ruta_csv = proyecto / "data" / "netflix_titles.csv"

    print("INICIANDO PIPELINE NETFLIX ETL")
    print("-----------------------------")

    # EXTRACT
    datos = extraer_datos(ruta_csv)

    # TRANSFORM
    datos = limpiar_datos_base(datos)
    datos = normalizar_duraciones(datos)

    contenido, pais, genero = crear_tablas_relacionales(
        datos
    )

    # VALIDATE
    validar_datos(contenido, pais, genero)

    print("-----------------------------")
    print("PIPELINE DE TRANSFORMACIÓN COMPLETADO")
    print("Contenido:", len(contenido))
    print("Países:", len(pais))
    print("Géneros:", len(genero))


if __name__ == "__main__":
    main()
'''

ruta_main = proyecto / "src" / "main.py"

ruta_main.write_text(
    codigo_main,
    encoding="utf-8"
)

print("ARCHIVO MAIN.PY CREADO")
print("---------------------")
print("Existe:", ruta_main.is_file())
print("Ubicación:", ruta_main)


In [ ]:

import subprocess
import sys
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_main = proyecto / "src" / "main.py"

print("PRUEBA INDEPENDIENTE DEL PIPELINE")
print("--------------------------------")

resultado = subprocess.run(
    [sys.executable, str(ruta_main)],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print(resultado.stdout)

if resultado.returncode == 0:
    print("ESTADO: CORRECTO")
else:
    print("ESTADO: ERROR")
    print(resultado.stderr)


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

contenido_env = """# Configuración de MySQL
MYSQL_HOST=127.0.0.1
MYSQL_PORT=3306
MYSQL_DATABASE=netflix_etl_dev
MYSQL_USER=netflix_etl
MYSQL_PASSWORD=
"""

ruta_env = proyecto / ".env.example"

ruta_env.write_text(
    contenido_env,
    encoding="utf-8"
)

print("CONFIGURACIÓN DE ENTORNO")
print("------------------------")
print("Archivo creado:", ruta_env.is_file())
print("Nombre:", ruta_env.name)
print("Contraseña incluida: NO")


In [ ]:

from pathlib import Path
from getpass import getpass

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_env = proyecto / ".env"

if ruta_env.exists():
    print("El archivo .env ya existe.")
    print("No se modificó su contenido.")
else:
    contraseña = getpass(
        "Introduce la contraseña del usuario netflix_etl: "
    )

    if not contraseña:
        print("Contraseña vacía. No se creó el archivo.")
    elif "\n" in contraseña or "\r" in contraseña:
        print("Contraseña no válida para este formato.")
    else:
        configuracion = (
            "MYSQL_HOST=127.0.0.1\n"
            "MYSQL_PORT=3306\n"
            "MYSQL_DATABASE=netflix_etl_dev\n"
            "MYSQL_USER=netflix_etl\n"
            f"MYSQL_PASSWORD={contraseña}\n"
        )

        ruta_env.write_text(
            configuracion,
            encoding="utf-8"
        )

        print("ARCHIVO PRIVADO CREADO")
        print("---------------------")
        print("Existe:", ruta_env.is_file())
        print("Contraseña mostrada: NO")

    del contraseña


In [ ]:


import sys
import subprocess
from importlib.metadata import version

resultado = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "python-dotenv"
    ],
    capture_output=True,
    text=True
)

print("INSTALACIÓN DE PYTHON-DOTENV")
print("---------------------------")

if resultado.returncode == 0:
    print("Estado: CORRECTO")
    print("Versión:", version("python-dotenv"))
else:
    print("Estado: ERROR")
    print(resultado.stderr)



In [ ]:

from pathlib import Path
import ast

proyecto = Path.home() / "netflix-etl-pipeline"

codigo_database = '''import os
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


def crear_conexion():
    proyecto = Path(__file__).resolve().parent.parent
    ruta_env = proyecto / ".env"

    if not ruta_env.is_file():
        raise FileNotFoundError(
            "No se encontró el archivo .env"
        )

    load_dotenv(ruta_env, override=True)

    variables = [
        "MYSQL_HOST",
        "MYSQL_PORT",
        "MYSQL_DATABASE",
        "MYSQL_USER",
        "MYSQL_PASSWORD"
    ]

    faltantes = [
        variable for variable in variables
        if not os.getenv(variable)
    ]

    if faltantes:
        raise ValueError(
            "Faltan variables de configuración: "
            + ", ".join(faltantes)
        )

    if os.getenv("MYSQL_DATABASE") != "netflix_etl_dev":
        raise ValueError(
            "Base de datos no autorizada"
        )

    url = URL.create(
        drivername="mysql+pymysql",
        username=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        host=os.getenv("MYSQL_HOST"),
        port=int(os.getenv("MYSQL_PORT")),
        database=os.getenv("MYSQL_DATABASE")
    )

    engine = create_engine(
        url,
        pool_pre_ping=True
    )

    with engine.connect() as conexion:
        base_actual = conexion.execute(
            text("SELECT DATABASE()")
        ).scalar_one()

        if base_actual != "netflix_etl_dev":
            raise ValueError(
                "La conexión apunta a una base no autorizada"
            )

    return engine
'''

# Comprobar sintaxis antes de guardar
ast.parse(codigo_database)

ruta_database = proyecto / "src" / "database.py"

ruta_database.write_text(
    codigo_database,
    encoding="utf-8"
)

print("ARCHIVO DATABASE.PY CREADO")
print("-------------------------")
print("Existe:", ruta_database.is_file())
print("Sintaxis Python: CORRECTA")
print("Contraseña incluida en el código: NO")


In [ ]:

import importlib
import database
from sqlalchemy import text

# Cargar el módulo actualizado
importlib.reload(database)

print("PRUEBA DE CONEXIÓN MYSQL")
print("-----------------------")

# Crear la conexión desde el archivo .env
engine_seguro = database.crear_conexion()

# Ejecutar una consulta de solo lectura
with engine_seguro.connect() as conexion:
    base_actual = conexion.execute(
        text("SELECT DATABASE()")
    ).scalar_one()

    version_mysql = conexion.execute(
        text("SELECT VERSION()")
    ).scalar_one()

print("Conexión: EXITOSA")
print("Base de datos:", base_actual)
print("Versión MySQL:", version_mysql)
print("Credenciales mostradas: NO")


In [ ]:

from pathlib import Path
import ast

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_main = proyecto / "src" / "main.py"

codigo_main = ruta_main.read_text(encoding="utf-8")

# Incorporar las importaciones necesarias
importaciones = (
    "from database import crear_conexion\n"
    "from load import cargar_datos\n"
)

if "from database import crear_conexion" not in codigo_main:
    codigo_main = importaciones + codigo_main

# Incorporar la etapa de carga al final del proceso
bloque_carga = '''
    # LOAD
    engine = crear_conexion()

    try:
        cargar_datos(
            engine,
            contenido,
            pais,
            genero
        )
    finally:
        engine.dispose()

    print("PIPELINE ETL FINALIZADO")
'''

if "# LOAD" not in codigo_main:
    codigo_main = codigo_main.replace(
        '\n\nif __name__ == "__main__":',
        '\n' + bloque_carga
        + '\n\nif __name__ == "__main__":'
    )

# Verificar la sintaxis antes de guardar
ast.parse(codigo_main)

ruta_main.write_text(
    codigo_main,
    encoding="utf-8"
)

print("MAIN.PY ACTUALIZADO")
print("-------------------")
print("Conexión MySQL integrada: SÍ")
print("Módulo de carga integrado: SÍ")
print("Sintaxis Python: CORRECTA")


In [ ]:

import subprocess
import sys
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_main = proyecto / "src" / "main.py"

print("PRUEBA FINAL DEL PIPELINE ETL")
print("----------------------------")

resultado = subprocess.run(
    [sys.executable, str(ruta_main)],
    cwd=proyecto,
    capture_output=True,
    text=True,
    timeout=120
)

print(resultado.stdout)

if resultado.returncode == 0:
    print("ESTADO FINAL: CORRECTO")
else:
    print("ESTADO FINAL: ERROR")
    print(resultado.stderr)


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_requirements = proyecto / "requirements.txt"

dependencias = [
    "pandas==3.0.3",
    "SQLAlchemy==2.0.51",
    "PyMySQL==2.2.8",
    "python-dotenv==1.2.1"
]

ruta_requirements.write_text(
    "\n".join(dependencias) + "\n",
    encoding="utf-8"
)

print("REQUIREMENTS.TXT ACTUALIZADO")
print("---------------------------")
print(ruta_requirements.read_text(encoding="utf-8"))
print("Total de dependencias:", len(dependencias))


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

readme = """# Netflix ETL Pipeline

Proyecto de ingeniería de datos desarrollado con Python y MySQL.

## Objetivo

Construir un pipeline ETL que extraiga información de un
dataset de Netflix, limpie y normalice los datos, valide
su integridad y los almacene en una base de datos relacional.

## Tecnologías

- Python
- Pandas
- MySQL
- SQLAlchemy
- PyMySQL
- python-dotenv
- Jupyter Notebook

## Arquitectura del pipeline

CSV de Netflix
    |
    v
Extract: lectura del archivo
    |
    v
Transform: limpieza y normalización
    |
    v
Validate: controles de calidad
    |
    v
Load: almacenamiento en MySQL

## Estructura del proyecto

```text
netflix-etl-pipeline/
├── data/
│   └── netflix_titles.csv
├── notebooks/
│   └── 01_netflix_pipeline_etl.ipynb
├── src/
│   ├── extract.py
│   ├── transform.py
│   ├── validate.py
│   ├── load.py
│   ├── database.py
│   └── main.py
├── sql/
├── .env.example
├── .gitignore
├── requirements.txt
└── README.md
```

## Modelo de datos

El pipeline genera tres tablas relacionales:

| Tabla | Registros |
|---|---:|
| contenido | 8.809 |
| contenido_pais | 10.845 |
| contenido_genero | 19.329 |

Total: 38.983 registros.

## Instalación

1. Clonar el repositorio.
2. Instalar las dependencias:

```bash
pip install -r requirements.txt
```

3. Crear un archivo `.env` utilizando `.env.example`
   como referencia.
4. Configurar una base de datos MySQL llamada
   `netflix_etl_dev` con las tres tablas necesarias.
5. Ejecutar el pipeline:

```bash
python src/main.py
```

## Validaciones

El proyecto incluye controles de:

- Identificadores duplicados.
- Valores obligatorios vacíos.
- Integridad referencial.
- Consistencia de las tablas.
- Prevención de cargas duplicadas cuando las tablas
  ya contienen registros.

## Estado del proyecto

Pipeline ETL local implementado y probado correctamente.

La carga actual es de tipo inicial: cuando las tablas
ya contienen registros, el programa omite la inserción.
No implementa todavía cargas incrementales.

## Próximas mejoras

- Automatización de ejecuciones.
- Registro de logs.
- Pruebas automatizadas.
- Implementación en Microsoft Azure.
"""

ruta_readme = proyecto / "README.md"

ruta_readme.write_text(
    readme,
    encoding="utf-8"
)

print("README.MD CREADO")
print("----------------")
print("Existe:", ruta_readme.is_file())
print("Ubicación:", ruta_readme)
print("Tamaño:", ruta_readme.stat().st_size, "bytes")


In [ ]:

from sqlalchemy import text

tablas = [
    "contenido",
    "contenido_pais",
    "contenido_genero"
]

print("ESTRUCTURA REAL DE MYSQL")
print("-----------------------")

with engine_seguro.connect() as conexion:
    for tabla in tablas:
        resultado = conexion.execute(
            text(f"SHOW CREATE TABLE `{tabla}`")
        ).fetchone()

        print(f"\nTABLA: {tabla}")
        print(resultado[1])
        print("-" * 50)

print("\nCONSULTA FINALIZADA")


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_sql = proyecto / "sql" / "create_tables.sql"

codigo_sql = """-- Netflix ETL Pipeline
-- Esquema de tablas para MySQL 8
-- Ejecutar en la base de datos netflix_etl_dev

CREATE TABLE IF NOT EXISTS contenido (
    show_id VARCHAR(10) NOT NULL,
    type VARCHAR(20) NOT NULL,
    title VARCHAR(500) NOT NULL,
    release_year SMALLINT DEFAULT NULL,
    rating VARCHAR(30) DEFAULT NULL,
    duration_value SMALLINT DEFAULT NULL,
    duration_unit VARCHAR(20) DEFAULT NULL,
    PRIMARY KEY (show_id)
) ENGINE=InnoDB
DEFAULT CHARSET=utf8mb4
COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS contenido_pais (
    show_id VARCHAR(10) NOT NULL,
    country VARCHAR(150) NOT NULL,
    PRIMARY KEY (show_id, country),
    CONSTRAINT contenido_pais_ibfk_1
        FOREIGN KEY (show_id)
        REFERENCES contenido(show_id)
) ENGINE=InnoDB
DEFAULT CHARSET=utf8mb4
COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS contenido_genero (
    show_id VARCHAR(10) NOT NULL,
    genre VARCHAR(150) NOT NULL,
    PRIMARY KEY (show_id, genre),
    CONSTRAINT contenido_genero_ibfk_1
        FOREIGN KEY (show_id)
        REFERENCES contenido(show_id)
) ENGINE=InnoDB
DEFAULT CHARSET=utf8mb4
COLLATE=utf8mb4_unicode_ci;
"""

ruta_sql.write_text(
    codigo_sql,
    encoding="utf-8"
)

print("ARCHIVO SQL CREADO")
print("-----------------")
print("Existe:", ruta_sql.is_file())
print("Ubicación:", ruta_sql)
print(
    "Tablas definidas:",
    codigo_sql.count("CREATE TABLE IF NOT EXISTS")
)


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

ruta_gitignore = proyecto / ".gitignore"
ruta_env = proyecto / ".env"
ruta_ejemplo = proyecto / ".env.example"

print("REVISIÓN DE SEGURIDAD PARA GITHUB")
print("--------------------------------")

if ruta_gitignore.is_file():
    reglas = ruta_gitignore.read_text(
        encoding="utf-8"
    ).splitlines()

    reglas = [
        linea.strip()
        for linea in reglas
        if linea.strip() and not linea.startswith("#")
    ]

    print("Archivo .gitignore: EXISTE")
    print("Regla para .env:", ".env" in reglas)
    print("Regla para .env.*:", ".env.*" in reglas)
    print(
        "Excepción para .env.example:",
        "!.env.example" in reglas
    )
else:
    print("Archivo .gitignore: NO EXISTE")

print("Archivo privado .env:", ruta_env.is_file())
print("Archivo .env.example:", ruta_ejemplo.is_file())

print("--------------------------------")
print("Revisión finalizada")


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

print("VERIFICACIÓN DE GIT")
print("-------------------")

try:
    resultado = subprocess.run(
        ["git", "--version"],
        cwd=proyecto,
        capture_output=True,
        text=True,
        timeout=15
    )

    if resultado.returncode == 0:
        print("Git instalado: SÍ")
        print("Versión:", resultado.stdout.strip())
    else:
        print("Git instalado: NO SE PUDO VERIFICAR")
        print("Detalle:", resultado.stderr.strip())

except FileNotFoundError:
    print("Git instalado: NO")
    print("Git no está disponible en el sistema.")

print("-------------------")
print("Verificación finalizada")


In [ ]:

from pathlib import Path

rutas_git = [
    Path("C:/Program Files/Git/cmd/git.exe"),
    Path("C:/Program Files/Git/bin/git.exe"),
    Path("C:/Program Files (x86)/Git/cmd/git.exe"),
    Path.home() / "AppData/Local/Programs/Git/cmd/git.exe"
]

print("BÚSQUEDA DE GIT EN WINDOWS")
print("-------------------------")

encontrado = False

for ruta in rutas_git:
    if ruta.is_file():
        print("Git encontrado en:", ruta)
        encontrado = True

if not encontrado:
    print("Git no se encontró en las rutas habituales.")

print("-------------------------")
print("Búsqueda finalizada")


In [ ]:

import os
import subprocess
from pathlib import Path

# Ubicación de Git en Windows
ruta_git = Path("C:/Program Files/Git/cmd")

# Agregar Git a las rutas de esta sesión de Jupyter
os.environ["PATH"] = (
    str(ruta_git)
    + os.pathsep
    + os.environ.get("PATH", "")
)

# Comprobar que funciona
resultado = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
)

print("VERIFICACIÓN DE GIT")
print("-------------------")
print("Estado:", "CORRECTO" if resultado.returncode == 0 else "ERROR")
print("Versión:", resultado.stdout.strip())


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

print("INICIALIZACIÓN DEL REPOSITORIO")
print("-----------------------------")

resultado = subprocess.run(
    ["git", "init"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

if resultado.returncode == 0:
    print("Estado: CORRECTO")
    print("Repositorio Git:", (proyecto / ".git").is_dir())
    print("Ubicación:", proyecto)
else:
    print("Estado: ERROR")
    print(resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

print("VERIFICACIÓN DE SEGURIDAD GIT")
print("----------------------------")

archivos = [
    ".env",
    ".env.example",
    "src/main.py",
    "requirements.txt"
]

for archivo in archivos:
    resultado = subprocess.run(
        ["git", "check-ignore", "-q", archivo],
        cwd=proyecto,
        capture_output=True,
        text=True
    )

    if resultado.returncode == 0:
        estado = "IGNORADO"
    elif resultado.returncode == 1:
        estado = "NO IGNORADO"
    else:
        estado = "ERROR"

    print(f"{archivo}: {estado}")

print("----------------------------")
print("Verificación finalizada")


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("ARCHIVOS DETECTADOS POR GIT")
print("--------------------------")

if resultado.returncode == 0:
    print(resultado.stdout or "No hay cambios pendientes.")
else:
    print("Error al consultar Git:")
    print(resultado.stderr)

print("--------------------------")
print("Revisión finalizada")


In [ ]:

import json
import re
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

ruta_notebook = (
    proyecto
    / "notebooks"
    / "01_netflix_pipeline_etl.ipynb"
)

with ruta_notebook.open("r", encoding="utf-8") as archivo:
    nb = json.load(archivo)

patrones = {
    "Posible contraseña literal":
        r"""(?i)\b(?:password|passwd|pwd|contraseña|clave)\s*[:=]\s*['"][^'"]+['"]""",

    "Posible credencial SQL":
        r"(?i)IDENTIFIED\s+BY",

    "URL MySQL con credenciales":
        r"(?i)mysql(?:\+pymysql)?://[^\s@]+@",

    "Posible token":
        r"(?i)\b(?:api_key|access_token|secret_key)\s*[:=]"
}

print("AUDITORÍA DEL NOTEBOOK")
print("----------------------")

hallazgos = 0
salidas = 0

for numero, celda in enumerate(nb["cells"], 1):
    contenido = "".join(celda.get("source", []))

    if celda["cell_type"] == "code":
        salidas += len(celda.get("outputs", []))

    for descripcion, patron in patrones.items():
        if re.search(patron, contenido):
            print(f"Celda {numero}: {descripcion}")
            hallazgos += 1

print("----------------------")
print("Posibles coincidencias:", hallazgos)
print("Salidas guardadas:", salidas)
print("Auditoría finalizada")


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_readme = proyecto / "README.md"

contenido = ruta_readme.read_text(encoding="utf-8")

seccion = """
## Fuente de los datos

- Dataset: Netflix Movies and TV Shows
- Autor: Shivam Bansal
- Plataforma: Kaggle
- Fuente: https://www.kaggle.com/datasets/shivamb/netflix-shows
- Licencia publicada: CC0 (Public Domain)

El archivo `data/netflix_titles.csv` se utiliza
con fines de aprendizaje y demostración de un
pipeline de ingeniería de datos.

Este proyecto es independiente y no está
afiliado oficialmente con Netflix.
"""

if "## Fuente de los datos" not in contenido:
    contenido += "\n" + seccion
    ruta_readme.write_text(contenido, encoding="utf-8")

print("DOCUMENTACIÓN ACTUALIZADA")
print("-------------------------")
print("Fuente Kaggle incluida:", "shivamb/netflix-shows" in contenido + seccion)
print("Licencia CC0 documentada:", "CC0" in contenido + seccion)
print("Archivo README.md:", ruta_readme.is_file())


In [ ]:

from pathlib import Path
import re

proyecto = Path.home() / "netflix-etl-pipeline"
carpeta_src = proyecto / "src"

patrones = {
    "Posible contraseña escrita directamente":
        r"""(?i)\b(?:password|passwd|pwd|contraseña)\s*=\s*['"][^'"]+['"]""",

    "Posible URL con credenciales":
        r"(?i)mysql(?:\+pymysql)?://[^\s@]+@",

    "Posible clave o token":
        r"""(?i)\b(?:api_key|access_token|secret_key)\s*=\s*['"][^'"]+['"]"""
}

print("AUDITORÍA DE ARCHIVOS PYTHON")
print("---------------------------")

archivos = sorted(carpeta_src.glob("*.py"))
hallazgos = 0

for archivo in archivos:
    codigo = archivo.read_text(encoding="utf-8")

    for descripcion, patron in patrones.items():
        for coincidencia in re.finditer(patron, codigo):
            linea = codigo.count(
                "\n", 0, coincidencia.start()
            ) + 1

            print(
                f"{archivo.name}, línea {linea}: "
                f"{descripcion}"
            )

            hallazgos += 1

print("---------------------------")
print("Archivos revisados:", len(archivos))
print("Coincidencias sospechosas:", hallazgos)
print("Auditoría finalizada")


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("REVISIÓN FINAL DEL REPOSITORIO")
print("-----------------------------")

if resultado.returncode == 0:
    archivos = resultado.stdout.splitlines()

    for archivo in archivos:
        print("-", archivo)

    privados = [
        archivo for archivo in archivos
        if Path(archivo).name == ".env"
    ]

    print("-----------------------------")
    print("Total de archivos:", len(archivos))
    print("Archivo .env incluido:", bool(privados))
    print(
        "Estado:",
        "CORRECTO" if not privados else "REVISAR"
    )
else:
    print("ERROR AL CONSULTAR GIT")
    print(resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "add", "."],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("PREPARACIÓN DE ARCHIVOS")
print("-----------------------")

if resultado.returncode == 0:
    print("Archivos agregados correctamente.")
else:
    print("Error al preparar archivos:")
    print(resultado.stderr)

print("-----------------------")

verificacion = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

archivos = verificacion.stdout.splitlines()

print("Archivos preparados:", len(archivos))
print("Archivo .env preparado:", ".env" in archivos)
print("Estado:", "CORRECTO" if resultado.returncode == 0 and verificacion.returncode == 0 and ".env" not in archivos else "REVISAR")


In [ ]:

import subprocess
import re
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "diff", "--cached", "--unified=0"],
    cwd=proyecto,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

patrones = {
    "Posible contraseña literal":
        r"""(?i)\b(?:password|passwd|pwd)\s*[:=]\s*['"][^'"]+['"]""",
    "Posible URL con credenciales":
        r"(?i)mysql(?:\+pymysql)?://[^\s@]+@",
    "Posible token":
        r"""(?i)\b(?:api_key|access_token|secret_key)\s*[:=]\s*['"][^'"]+['"]"""
}

print("REVISIÓN DE SEGURIDAD DEL COMMIT")
print("--------------------------------")

hallazgos = 0

if resultado.returncode == 0:
    for nombre, patron in patrones.items():
        coincidencias = re.findall(patron, resultado.stdout)
        if coincidencias:
            print(f"{nombre}: {len(coincidencias)} coincidencia(s)")
            hallazgos += len(coincidencias)

    print("Coincidencias sospechosas:", hallazgos)
    print("Estado:", "CORRECTO" if hallazgos == 0 else "REVISAR")
else:
    print("Error al revisar Git:")
    print(resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    [
        "git", "commit",
        "-m", "Initial commit: Netflix ETL pipeline"
    ],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("PRIMER COMMIT DEL PROYECTO")
print("-------------------------")

if resultado.returncode == 0:
    print("COMMIT CREADO CORRECTAMENTE")
    print(resultado.stdout)
else:
    print("EL COMMIT NO SE PUDO CREAR")
    print(resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    [
        "git", "commit",
        "-m", "Initial commit: Netflix ETL pipeline"
    ],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("PRIMER COMMIT DEL PROYECTO")
print("-------------------------")

if resultado.returncode == 0:
    print("COMMIT CREADO CORRECTAMENTE")
    print(resultado.stdout)
else:
    print("ERROR AL CREAR EL COMMIT")
    print(resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "branch", "-M", "main"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

verificacion = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("CONFIGURACIÓN DE RAMA PRINCIPAL")
print("--------------------------------")

if resultado.returncode == 0:
    print("Rama actual:", verificacion.stdout.strip())
    print("Estado: CORRECTO")
else:
    print("Error:", resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

estado = subprocess.run(
    ["git", "status", "--short"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

rama = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

commit = subprocess.run(
    ["git", "log", "-1", "--format=%h %s"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("ESTADO FINAL DEL REPOSITORIO")
print("----------------------------")
print("Rama:", rama.stdout.strip())
print("Último commit:", commit.stdout.strip())
print(
    "Archivos pendientes:",
    estado.stdout.strip() or "Ninguno"
)

correcto = (
    estado.returncode == 0
    and rama.returncode == 0
    and commit.returncode == 0
    and rama.stdout.strip() == "main"
    and not estado.stdout.strip()
)

print("Estado:", "CORRECTO" if correcto else "REVISAR")


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "commit", "--amend", "--no-edit", "--reset-author"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("ACTUALIZACIÓN DEL PRIMER COMMIT")
print("--------------------------------")

if resultado.returncode == 0:
    print("COMMIT ACTUALIZADO CORRECTAMENTE")

    verificacion = subprocess.run(
        [
            "git", "log", "-1",
            "--format=Autor: %an%nCorreo: %ae%nCommit: %h"
        ],
        cwd=proyecto,
        capture_output=True,
        text=True
    )

    print(verificacion.stdout)
else:
    print("ERROR AL ACTUALIZAR EL COMMIT")
    print(resultado.stderr)


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    [
        "git", "log", "-1",
        "--format=Autor: %an <%ae>%nCommitter: %cn <%ce>%nCommit: %h"
    ],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("VERIFICACIÓN DE PRIVACIDAD")
print("-------------------------")

if resultado.returncode == 0:
    print(resultado.stdout)

    correos = [
        linea.split("<", 1)[1].split(">", 1)[0]
        for linea in resultado.stdout.splitlines()
        if linea.startswith(("Autor:", "Committer:"))
    ]

    correcto = (
        len(correos) == 2
        and all(
            correo.endswith("@users.noreply.github.com")
            for correo in correos
        )
    )

    print("Estado:", "CORRECTO" if correcto else "REVISAR")
else:
    print("ERROR:", resultado.stderr)


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_env = proyecto / ".env"

usuario = None

for linea in ruta_env.read_text(encoding="utf-8").splitlines():
    if linea.strip().startswith("DB_USER="):
        usuario = linea.split("=", 1)[1].strip().strip('"').strip("'")
        break

print("VERIFICACIÓN DEL USUARIO MYSQL")
print("------------------------------")
print("Usuario:", usuario if usuario else "No encontrado")
print("Base de datos del proyecto: netflix_etl_dev")
print("Contraseña: OCULTA")


In [ ]:

from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
ruta_env = proyecto / ".env"

print("VARIABLES CONFIGURADAS EN .env")
print("------------------------------")

if ruta_env.exists():
    for linea in ruta_env.read_text(encoding="utf-8").splitlines():
        linea = linea.strip()

        if not linea or linea.startswith("#") or "=" not in linea:
            continue

        nombre = linea.split("=", 1)[0].strip()

        print(nombre, "= [VALOR OCULTO]")
else:
    print("No se encontró el archivo .env")

print("------------------------------")
print("Revisión finalizada")


In [ ]:

from pathlib import Path
from dotenv import dotenv_values

proyecto = Path.home() / "netflix-etl-pipeline"

configuracion = dotenv_values(proyecto / ".env")

usuario = configuracion.get("MYSQL_USER")

print("IDENTIFICACIÓN DEL USUARIO MYSQL")
print("--------------------------------")
print("Usuario:", usuario or "No encontrado")
print("Contraseña: OCULTA")
print("Estado:", "CORRECTO" if usuario else "REVISAR")


In [ ]:

import sys
from pathlib import Path
from sqlalchemy import text

proyecto = Path.home() / "netflix-etl-pipeline"
sys.path.insert(0, str(proyecto / "src"))

from database import crear_conexion

engine = crear_conexion()

with engine.connect() as conexion:
    resultado = conexion.execute(
        text("""
            SELECT
                CURRENT_USER() AS cuenta_mysql,
                DATABASE() AS base_datos
        """)
    ).mappings().one()

print("VERIFICACIÓN DE CUENTA MYSQL")
print("----------------------------")
print("Cuenta autenticada:", resultado["cuenta_mysql"])
print("Base de datos:", resultado["base_datos"])
print("Estado: CONSULTA COMPLETADA")

engine.dispose()


In [ ]:

import subprocess
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

url_github = "https://github.com/Joha0112/netflix-etl-pipeline.git"

# Comprobar si ya existe una conexión llamada origin
consulta = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

# Crear o actualizar la conexión
comando = (
    ["git", "remote", "set-url", "origin", url_github]
    if consulta.returncode == 0
    else ["git", "remote", "add", "origin", url_github]
)

resultado = subprocess.run(
    comando,
    cwd=proyecto,
    capture_output=True,
    text=True
)

# Verificar la configuración
verificacion = subprocess.run(
    ["git", "remote", "-v"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("CONEXIÓN CON GITHUB")
print("------------------")

if resultado.returncode == 0:
    print(verificacion.stdout)
    print("Estado: CORRECTO")
else:
    print("Estado: ERROR")
    print(resultado.stderr)


In [ ]:


import subprocess
import os
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

entorno = os.environ.copy()
entorno["GIT_TERMINAL_PROMPT"] = "0"
entorno["GCM_INTERACTIVE"] = "never"

print("DIAGNÓSTICO DE CONEXIÓN CON GITHUB")
print("----------------------------------")

try:
    resultado = subprocess.run(
        ["git", "ls-remote", "--heads", "origin"],
        cwd=proyecto,
        capture_output=True,
        text=True,
        timeout=15,
        env=entorno
    )

    if resultado.returncode == 0:
        print("Conexión: CORRECTA")
        print(
            "Ramas remotas:",
            resultado.stdout.strip() or "Ninguna"
        )
    else:
        print("Conexión: REVISAR")
        print(resultado.stderr)

except subprocess.TimeoutExpired:
    print("Conexión: TIEMPO DE ESPERA AGOTADO")
    print("GitHub no respondió dentro de 15 segundos.")

print("----------------------------------")
print("Diagnóstico finalizado")



In [ ]:


import os
import subprocess
from pathlib import Path

ruta_git = r"C:\Program Files\Git\cmd"

os.environ["PATH"] = ruta_git + os.pathsep + os.environ.get("PATH", "")

proyecto = Path.home() / "netflix-etl-pipeline"

resultado = subprocess.run(
    ["git", "--version"],
    cwd=proyecto,
    capture_output=True,
    text=True
)

print("VERIFICACIÓN DE GIT")
print("-------------------")

if resultado.returncode == 0:
    print("Git encontrado correctamente")
    print("Versión:", resultado.stdout.strip())
    print("Estado: CORRECTO")
else:
    print("Estado: REVISAR")
    print(resultado.stderr)



In [ ]:

import subprocess
import os
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"

entorno = os.environ.copy()
entorno["GIT_TERMINAL_PROMPT"] = "0"
entorno["GCM_INTERACTIVE"] = "never"

print("VERIFICACIÓN DEL REPOSITORIO REMOTO")
print("----------------------------------")

try:
    resultado = subprocess.run(
        ["git", "ls-remote", "--heads", "origin"],
        cwd=proyecto,
        capture_output=True,
        text=True,
        timeout=20,
        env=entorno
    )

    if resultado.returncode == 0:
        if resultado.stdout.strip():
            print("GitHub tiene ramas publicadas:")
            print(resultado.stdout.strip())
            print("Estado: REVISAR")
        else:
            print("GitHub todavía no tiene ramas publicadas.")
            print("Estado: PENDIENTE DE PUBLICACIÓN")
    else:
        print("Estado: ERROR")
        print(resultado.stderr)

except subprocess.TimeoutExpired:
    print("Estado: TIEMPO DE ESPERA AGOTADO")

print("----------------------------------")
print("Verificación finalizada")


In [ ]:
import sys, importlib
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
sys.path.insert(0, str(proyecto / "src"))

import extract, transform, validate
for modulo in (extract, transform, validate):
    importlib.reload(modulo)

datos = extract.extraer_datos(proyecto / "data" / "netflix_titles.csv")
datos = transform.limpiar_datos_base(datos)

In [ ]:
print(extract.__file__)
print("Tiene eliminar_duplicados:", hasattr(transform, "eliminar_duplicados"))

In [ ]:
import sys, importlib
from pathlib import Path

proyecto = Path.home() / "netflix-etl-pipeline"
sys.path.insert(0, str(proyecto / "src"))

import extract, transform, validate
for modulo in (extract, transform, validate):
    importlib.reload(modulo)

print("Tiene eliminar_duplicados:", hasattr(transform, "eliminar_duplicados"))
print()

datos = extract.extraer_datos(proyecto / "data" / "netflix_titles.csv")
datos = transform.limpiar_datos_base(datos)

In [ ]:
import subprocess, sys

resultado = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
    cwd=proyecto,
    capture_output=True,
    text=True,
)

print(resultado.stderr[-2000:])

In [ ]:
import subprocess, sys

print(sorted(p.name for p in (proyecto / "tests").iterdir() if p.suffix == ".py"))
print()

resultado = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
    cwd=proyecto,
    capture_output=True,
    text=True,
)

print(resultado.stderr[-2000:])

In [ ]:
from sqlalchemy import text
import database

engine = database.crear_conexion()

with engine.begin() as conexion:
    for tabla in ["contenido_pais", "contenido_genero", "contenido"]:
        conexion.execute(text(f"DELETE FROM {tabla}"))

engine.dispose()
print("Tablas vaciadas")

In [ ]:
resultado = subprocess.run(
    [sys.executable, "src/main.py"],
    cwd=proyecto,
    capture_output=True,
    text=True,
)

print(resultado.stdout[-1500:])
print(resultado.stderr[-1500:])

In [ ]:
import pandas as pd
from sqlalchemy import text
import database

engine = database.crear_conexion()

with engine.connect() as conexion:
    conteos = {
        tabla: conexion.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar_one()
        for tabla in ["contenido", "contenido_pais", "contenido_genero"]
    }
    titulos = pd.read_sql(text("SELECT show_id, title FROM contenido"), conexion)

engine.dispose()

print(conteos)
print("Títulos con codificación dañada:", titulos["title"].str.contains("Ã|Â").sum())
print("Duplicados presentes:", titulos["show_id"].isin(["s5965", "s5966", "s5967", "s6530"]).sum())
print()
print(titulos[titulos["show_id"].isin(["s58", "s86", "s141", "s289"])].to_string(index=False))